In [ ]:
import pandas as pd

df = pd.read_csv("../adult_group_02.csv")  # the dataset is in the repo root

print(df.shape) # rows, columns
print(df.dtypes) # Python types
print(df.isna().sum()) # missing values per column
print(df.nunique()) # unique values per column
print((df["age"] < 0).sum()) # impossible ages
print(df["gender"].value_counts(dropna=False)) # typos + missing
print((df["marital-status"] == "?").sum()) # hidden missing
print(pd.crosstab(df["tax_category"], df["income"])) # proves leakage
print(pd.crosstab(df["workclass"], df["workclass_id"])) # proves duplicate


**Data dictionary** — 40,190 people, 14 columns.

| Column | Meaning | Type | Notes |
|---|---|---|---|
| age | Age of the person | Number | Ages go from 17 to 90. 201 ages are wrong (−10, −5), so we treat them as missing. |
| workclass | Who the person works for (private, government, …) | Category | 8 groups. "Without-pay" and "Never-worked" are very rare. |
| education | Highest education level | Category with order | 16 levels, from Preschool to Doctorate. |
| marital-status | Married, divorced, single, … | Category | 1,206 values are "?", which means missing. |
| occupation | Type of job | Category | 14 jobs. Nothing missing. |
| race | Race of the person | Category | 5 groups. Most people (83%) are White. Sensitive data, so be careful. |
| gender | Male or female | Two values | Has typos ("Mle", "Femle", "male") and 1,206 missing. Sensitive data, so be careful. |
| capital-gain | Money earned from investments | Number | Most people have 0. 242 people have 99,999, which is a maximum value, not a real amount. |
| capital-loss | Money lost on investments | Number | Most people have 0. |
| hours-per-week | Hours worked per week | Number | 1,607 missing. Goes from 1 to 99. |
| income | Earns more than \$50K a year, or not | Two values — **what we predict** | 74% earn ≤50K, 26% earn >50K. |
| workclass_id | Number code for workclass | Category (stored as a number) | Exact copy of workclass, so we remove it. |
| tax_category | Tax group: lower or higher | Two values | Always matches income, so it gives away the answer. We remove it. |
| survey_program | Name of the survey | Same value everywhere | Always "Census", so it tells us nothing. We remove it. |

## 0.1 Problem & Evaluation Design

### Alternative targets

| Other thing to predict | What we predict | Why useful |
|---|---|---|
| hours-per-week | How many hours a person works each week (a number) | Companies can plan their staff better |
| occupation | What kind of job a person has (one of 14 groups) | Job services can suggest jobs that fit the person |

### Final problem definition

**Business problem:** A bank has a new expensive credit card for people who earn a lot. The bank does not know how much people earn. Sending an offer costs money, so the bank cannot send it to everyone.

**Decision to support:** Do we send this person the offer? Yes or no.

**Unit of observation:** One row is one person.

**Prediction moment:** When the bank knows the person's basic info (age, job, education, hours worked, …), before sending offers.

**Success:** The model finds most of the people who earn more than 50K, and does not pick too many people who earn less. It must do better than the simple rule and the Dummy model.

**Target:** `income` — does the person earn more than 50K dollars a year? (yes = >50K, no = ≤50K). About 1 in 4 people earn more than 50K.

**Task:** Classification (sort people into two groups)

**Information not used:** We remove three columns before any modelling. `tax_category` always matches `income` (lower = ≤50K, higher = >50K, no exception in 40,190 rows), so it is the answer in another form and the bank would not know it at the prediction moment (target leakage). `workclass_id` is only a number code for `workclass` (the same information twice). `survey_program` is "Census" in every row (no information).

### Rule-based baseline

**Rule:** If the person is married → guess "earns more than 50K". If not → guess "earns 50K or less".

**Why this is useful:** Married people earn more than 50K much more often (46%) than people who never married (6%). The rule uses only one clue. Our ML models use all clues together (job, education, hours, …), so they should do better. If they don't, the ML is not worth it.

### Evaluation metrics

**Primary metric:** F1 score of the >50K class.

**Why appropriate:**

- Accuracy is misleading here: only 26% of the people earn more than 50K, so a model that always says "≤50K" is right 74% of the time but finds nobody.
- The bank can make two mistakes, and both cost something:
  - **False negative:** a person who earns >50K gets no offer → a lost customer. **Recall** measures this (of all >50K people, how many did we find?).
  - **False positive:** a person who earns ≤50K gets an offer → wasted cost. **Precision** measures this (of the people we picked, how many really earn >50K?).
- This is exactly our definition of success: find most of the >50K people *and* not pick too many others. We have no cost figures that show one mistake is clearly worse, so we weigh them equally.
- F1 is the harmonic mean of precision and recall, so it is only high when both are high. Sending the offer to everyone gives recall 100% but precision 26%, which is an F1 of only 0.41.
- If the bank later shows that one mistake costs more, the same reasoning leads to Fβ (β > 1 puts more weight on recall, β < 1 on precision).

**Supporting metric(s):**

- **Precision** and **recall**: show which of the two mistakes a model makes more.
- **PR-AUC** (average precision): precision and recall over all decision thresholds instead of only the default 0.5. It only looks at the >50K class, so it suits an imbalanced target. A model with no skill scores about 0.26 (the share of >50K).
- **ROC-AUC**: how well the model ranks >50K people above ≤50K people, over all thresholds. A model with no skill scores 0.5.
- **Accuracy**: only to show why it fools us here (the Dummy gets 74%).

Every baseline, model and tuning run is scored with this same set (the `SCORING` dictionary in S1.2 below); F1 decides.

## S1.2 — Frozen scoring — Artiom

One `SCORING` dictionary for the whole project. Stage 2 copies it unchanged into the contracts cell; every `cross_validate` and every search uses it, and the searches use `refit="f1"`.

In [ ]:
# These scorers expect the target as numbers: 1 = ">50K" (the positive class), 0 = "<=50K".
# With the text labels, f1 / precision / recall / average_precision come back as NaN
# (scikit-learn looks for pos_label=1), so S2.2 encodes the target before the split.
# average_precision and roc_auc are computed from scores, so every estimator, also the
# rule baseline, needs predict_proba (or decision_function).
SCORING = {
    "f1": "f1",                                # primary metric
    "precision": "precision",
    "recall": "recall",
    "average_precision": "average_precision",  # PR-AUC
    "roc_auc": "roc_auc",
    "accuracy": "accuracy",                    # only to show the Dummy trap
}